# 🧠 ClipFactory AI Director — Phase 3: Intelligence
> The ultimate AI-powered video repurposing engine. Automatically detect personas, remove fluff, and generate story-complete Shorts for the Whop Content Rewards program.

**Runtime:** Make sure you're on **T4 GPU** (Runtime > Change runtime type > T4 GPU) for LLaMA 3 acceleration.

### What's New in Phase 3
- **AI Video Intelligence:** Auto-detects genre, tone, and audience.
- **Smart Fluff Removal:** Multi-segment stitching for tighter storytelling.
- **Model Upgrade:** Defaulting to **LLaMA 3 8B** for superior logic.
- **SaaS 2.0 UI:** Glassmorphism dashboard with duration previews.

### Steps
1. Run **Cell 0** to mount Google Drive (models persist across sessions)
2. Run **Cell 1** once per session (installs everything, ~3 min)
3. Run **Cell 2** to launch the app — open the public URL it prints


In [ ]:
# ── OPTIONAL: Mount Google Drive Without the Python Auth Wall ───────────────
# To avoid the massive verification popup, DO NOT use python to mount.
# Instead, click the 📁 Folder icon on the far left sidebar of Colab,
# then click the 'Mount Drive' icon (a folder with the Google Drive logo).
# This uses your browser's active session securely without a python popup.
import os
if os.path.exists('/content/drive/MyDrive'):
    print('✅ Drive successfully detected.')
else:
    print('⚠️ Drive not mounted. Your files will be saved to temporary Colab storage.')

In [ ]:
# ── CELL 1: Setup (run once per session, ~3 min) ───────────────────────────
import os, subprocess, sys

REPO_DIR = '/content/clip-factory'

# Clone or update the repo
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/NiL4gh/clip-factory',
                    REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print('Repo up to date.')

# Install Python dependencies
!pip install -q -r {REPO_DIR}/requirements.txt

# Install llama-cpp-python from prebuilt CUDA wheel (skips 10-min compile)
!pip install llama-cpp-python --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

# Install Deno for yt-dlp JS challenge solver (fixes YouTube bot-check)
!curl -fsSL https://deno.land/install.sh | sh
os.environ['PATH'] = os.path.expanduser('~/.deno/bin') + ':' + os.environ['PATH']

# Install fonts for captions
!apt-get install -q -y fonts-liberation 2>/dev/null || true

sys.path.insert(0, REPO_DIR)
print('✅ Ready to launch.')

In [ ]:
# ── CELL 2: Launch ClipFactory.ai ────────────────────────
import os, sys, subprocess

# Check for GPU
try:
    subprocess.check_output('nvidia-smi')
    print('✅ GPU Detected. LLaMA 3 will run with acceleration.')
except:
    print('❌ WARNING: No GPU detected! The AI Director will be extremely slow on CPU.')

REPO_DIR = '/content/clip-factory'
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Auto-detect Drive for persistence
if os.path.exists('/content/drive/MyDrive'):
    BASE = '/content/drive/MyDrive/clip_factory'
    print('✅ Saving models and projects to Google Drive.')
else:
    BASE = '/content/clip_factory'
    print('⚠️ Using local ephemeral storage.')

os.environ['LLM_DIR']      = f'{BASE}/models/llm'
os.environ['WHISPER_DIR']  = f'{BASE}/models/whisper'
os.environ['OUTPUT_DIR']   = f'{BASE}/output'
os.environ['PROJECTS_DIR'] = f'{BASE}/projects'

for d in [os.environ['LLM_DIR'], os.environ['WHISPER_DIR'],
          os.environ['OUTPUT_DIR'], os.environ['PROJECTS_DIR']]:
    os.makedirs(d, exist_ok=True)

!python app.py
